# Chapter 13 · Sampling and Estimation: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, the list of schoolchildren of a municipality and an oral-health survey, through the secretary's ten questions. **Part B** gives the written solution of every chapter exercise (statements are in the book) and, for the exercises whose solution needs a calculation, its code and result. Run the cells in order.

## Part A. The case

**Data.** `data/schoolchildren.csv`: 14,299 rows, one per enrolled child, and four columns: `child_id`, `school` (S01 to S48), `school_type` (`private` or `public`), which the education office's list holds, and `dmft`, which in real life is known only after the examination and is included here for every child so that each method can be checked against the truth. Each cell carries the number of the book section it belongs to.

### 13.1 Introduction: the data

In [1]:
# 1. Read the file: one row is one enrolled child (dmft is known in real life only after the examination)
import numpy as np
import pandas as pd
from math import sqrt
from scipy import stats

df = pd.read_csv("data/schoolchildren.csv")

# 2. Count rows and columns
rows, columns = df.shape
print("The list has", rows, "rows (one per child) and", columns, "columns.")
df.head()                                      # The first five rows

The list has 14299 rows (one per child) and 4 columns.


,child_id,school,school_type,dmft
0,K00001,S01,private,5
1,K00002,S01,private,2
2,K00003,S01,private,3
3,K00004,S01,private,0
4,K00005,S01,private,2


### 13.2 Whom does the survey describe?

*Whom does the survey describe, and does the way the children are chosen change what the result means?*

In [2]:
# 1. The whole list: population mean, standard deviation, and share free of decay
dmft = df["dmft"].values
N = len(dmft)
mu = dmft.mean()
sigma = dmft.std(ddof=0)
print("Population: mean", round(mu, 3), "; standard deviation", round(sigma, 3), "; free of decay:", round(100 * (dmft == 0).mean(), 1), "%;", N, "children")

# 2. By type of school
display(df.groupby("school_type")["dmft"].agg(["size", "mean", "std"]).round(3))

# 3. A convenience sample: the first 600 children of the list
first600 = df.iloc[:600]
print("Convenience sample: mean", round(first600["dmft"].mean(), 3), "; free of decay:", round(100 * (first600["dmft"] == 0).mean(), 1), "%; schools:", first600["school"].nunique())

Population: mean 1.87 ; standard deviation 2.389 ; free of decay: 35.7 %; 14299 children


,size,mean,std
school_type,,,
private,3448,0.997,1.416
public,10851,2.148,2.562


Convenience sample: mean 1.295 ; free of decay: 44.0 %; schools: 3


### 13.3 Drawing the children at random

*How are 600 children drawn at random, and by what other rules can a probability sample be drawn?*

In [3]:
# 1. One simple random sample of 600 children (seed 13)
srs = df.sample(n=600, random_state=13)
print("Simple random sample: mean", round(srs["dmft"].mean(), 3), "; schools:", srs["school"].nunique(), "; private:", round(100 * (srs["school_type"] == "private").mean(), 1), "%")

# 2. Systematic sample: every 24th child from a random start, repeated 5,000 times
rng = np.random.default_rng(13)
systematic_means = []
for repetition in range(5000):
    start = rng.integers(0, 24)
    systematic_means.append(dmft[np.arange(start, N, 24)].mean())
srs_se = sigma / sqrt(600) * sqrt((N - 600) / (N - 1))
print("Systematic: standard deviation of the means", round(np.std(systematic_means), 4), "; simple random sample standard error", round(srs_se, 4))

Simple random sample: mean 1.847 ; schools: 48 ; private: 24.3 %
Systematic: standard deviation of the means 0.0841 ; simple random sample standard error 0.0954


### 13.4 How much does the average change from sample to sample?

*How much would the average dmft change from one sample to another?*

In [4]:
# 1. 2,000 simple random samples of 600 children: the sampling distribution of the mean
rng = np.random.default_rng(13)
means600 = []
for repetition in range(2000):
    means600.append(dmft[rng.choice(N, 600, replace=False)].mean())
means600 = np.array(means600)
print("Means: mean", round(means600.mean(), 3), "; standard deviation of the means", round(means600.std(), 4))

# 2. Standard error by the formula, without and with the finite population correction
fpc = sqrt((N - 600) / (N - 1))
print("Standard error: sigma/sqrt(n) =", round(sigma / sqrt(600), 4), "; with the correction", round(sigma / sqrt(600) * fpc, 4), "; correction factor", round(fpc, 3))

# 3. Variance dividing by n and by n - 1, in 5,000 samples of 10 children
rng = np.random.default_rng(13)
variances_n = []
variances_n1 = []
for repetition in range(5000):
    small = dmft[rng.choice(N, 10, replace=False)]
    variances_n.append(small.var(ddof=0))
    variances_n1.append(small.var(ddof=1))
print("Population variance", round(sigma ** 2, 3), "; dividing by n", round(np.mean(variances_n), 3), "; dividing by n - 1", round(np.mean(variances_n1), 3))

# 4. Mean of the first n children in 20 random orders of the list
rng = np.random.default_rng(13)
paths = []
for order in range(20):
    paths.append(np.cumsum(dmft[rng.permutation(N)]) / np.arange(1, N + 1))
paths = np.array(paths)
for n_children in (30, 600, 3000):
    column = paths[:, n_children - 1]
    print("Number of children:", n_children, "; the 20 means range from", round(column.min(), 2), "up to", round(column.max(), 2))

# 5. Half-width of the population mean plus and minus 1.96 standard errors
for n_children in (600, 3000):
    half_width = 1.96 * sigma / sqrt(n_children) * sqrt((N - n_children) / (N - 1))
    print("Number of children:", n_children, "; band half-width (1.96 standard errors):", round(half_width, 2))

Means: mean 1.873 ; standard deviation of the means 0.093
Standard error: sigma/sqrt(n) = 0.0975 ; with the correction 0.0954 ; correction factor 0.979
Population variance 5.705 ; dividing by n 5.133 ; dividing by n - 1 5.704
Number of children: 30 ; the 20 means range from 1.3 up to 2.63
Number of children: 600 ; the 20 means range from 1.72 up to 2.04
Number of children: 3000 ; the 20 means range from 1.82 up to 1.95
Number of children: 600 ; band half-width (1.96 standard errors): 0.19
Number of children: 3000 ; band half-width (1.96 standard errors): 0.08


### 13.5 What shape does the variation of the average take?

*Do the averages of the samples spread in a bell shape, even though the dmft of the children does not have that shape?*

In [5]:
# 1. Sample means for samples of 5, 30, and 600 children (2,000 samples each)
for n_children in (5, 30, 600):
    rng = np.random.default_rng(13)
    sample_means = []
    for repetition in range(2000):
        sample_means.append(dmft[rng.choice(N, n_children, replace=False)].mean())
    print("Samples of", n_children, ": skewness of the means", round(stats.skew(sample_means), 2), "; standard deviation", round(np.std(sample_means), 3), "; sigma/sqrt(n):", round(sigma / sqrt(n_children), 3))

Samples of 5 : skewness of the means 0.84 ; standard deviation 1.065 ; sigma/sqrt(n): 1.068
Samples of 30 : skewness of the means 0.41 ; standard deviation 0.452 ; sigma/sqrt(n): 0.436
Samples of 600 : skewness of the means 0.05 ; standard deviation 0.093 ; sigma/sqrt(n): 0.098


### 13.6 The average dmft and a confidence interval

*What is the average dmft in the municipality, and how sure are we?*

In [6]:
# 1. The sample of 600: mean, standard deviation, and standard error
x = srs["dmft"].values
n = len(x)
xbar = x.mean()
s = x.std(ddof=1)
se = s / sqrt(n)
print("Sample: mean", round(xbar, 3), "; standard deviation", round(s, 3), "; standard error", round(se, 4))

# 2. 95% confidence interval for the mean with the t multiple
t_multiple = stats.t.ppf(0.975, n - 1)
low = xbar - t_multiple * se
high = xbar + t_multiple * se
print("t multiple:", round(t_multiple, 3), "; 95% CI:", round(low, 3), "to", round(high, 3), "; contains the population mean:", low <= mu <= high)

# 3. Coverage of the interval in 2,000 samples of 600 children
rng = np.random.default_rng(13)
hits = 0
for repetition in range(2000):
    sample = dmft[rng.choice(N, 600, replace=False)]
    margin = t_multiple * sample.std(ddof=1) / sqrt(600)
    if abs(sample.mean() - mu) <= margin:
        hits = hits + 1
print("Coverage with 600 children:", round(100 * hits / 2000, 1), "%")

# 4. Samples of 15 children: intervals with 1.96 and with the t multiple, 5,000 samples
rng = np.random.default_rng(13)
hits_z = 0
hits_t = 0
for repetition in range(5000):
    sample = dmft[rng.choice(N, 15, replace=False)]
    se15 = sample.std(ddof=1) / sqrt(15)
    if abs(sample.mean() - mu) <= 1.96 * se15:
        hits_z = hits_z + 1
    if abs(sample.mean() - mu) <= stats.t.ppf(0.975, 14) * se15:
        hits_t = hits_t + 1
print("Coverage with 15 children: 1.96", round(100 * hits_z / 5000, 1), "%; t multiple", round(100 * hits_t / 5000, 1), "%")

Sample: mean 1.847 ; standard deviation 2.373 ; standard error 0.0969
t multiple: 1.964 ; 95% CI: 1.656 to 2.037 ; contains the population mean: True
Coverage with 600 children: 95.5 %


Coverage with 15 children: 1.96 88.4 %; t multiple 90.0 %


### 13.7 Shares of children and the Wald and Wilson intervals

*What share of the children is free of decay, or has severe decay, and how sure are we?*

In [7]:
# 1. Shares in the sample of 600: free of decay and dmft of 8 or more
z = 1.959964
for label, count in (("Free of decay:", int((x == 0).sum())), ("Severe (dmft 8 or more):", int((x >= 8).sum()))):
    share = count / n
    se_share = sqrt(share * (1 - share) / n)

    # 2. Wald and Wilson 95% intervals
    wald_low = share - z * se_share
    wald_high = share + z * se_share
    centre = (share + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * sqrt(share * (1 - share) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    print(label, count, "of", n, "; share", round(100 * share, 1), "%; standard error", round(se_share, 4))
    print("Wald:", round(100 * wald_low, 1), "to", round(100 * wald_high, 1), "; Wilson:", round(100 * (centre - half), 1), "to", round(100 * (centre + half), 1))

# 3. Exact coverage of both intervals for samples of 100 children drawn without replacement (hypergeometric) when the true share is that of severe decay
p_true = (dmft >= 8).mean()
counts = np.arange(0, 101)
shares = counts / 100
wald_l = shares - z * np.sqrt(shares * (1 - shares) / 100)
wald_h = shares + z * np.sqrt(shares * (1 - shares) / 100)
wilson_c = (shares + z ** 2 / 200) / (1 + z ** 2 / 100)
wilson_h = z * np.sqrt(shares * (1 - shares) / 100 + z ** 2 / 40000) / (1 + z ** 2 / 100)
probabilities = stats.hypergeom.pmf(counts, N, int((dmft >= 8).sum()), 100)
print("True share:", round(100 * p_true, 2), "%; coverage Wald:", round(100 * probabilities[(wald_l <= p_true) & (p_true <= wald_h)].sum(), 1), "%; Wilson:", round(100 * probabilities[(wilson_c - wilson_h <= p_true) & (p_true <= wilson_c + wilson_h)].sum(), 1), "%")

Free of decay: 229 of 600 ; share 38.2 %; standard error 0.0198
Wald: 34.3 to 42.1 ; Wilson: 34.4 to 42.1
Severe (dmft 8 or more): 27 of 600 ; share 4.5 %; standard error 0.0085
Wald: 2.8 to 6.2 ; Wilson: 3.1 to 6.5
True share: 3.58 %; coverage Wald: 87.5 %; Wilson: 97.3 %


### 13.8 Stratifying by type of school

*Should private and public schools be sampled separately?*

In [8]:
# 1. The two strata: sizes, weights, means, and standard deviations
public = df[df["school_type"] == "public"]["dmft"].values
private = df[df["school_type"] == "private"]["dmft"].values
sizes = np.array([len(public), len(private)])
weights = sizes / N
sds = np.array([public.std(ddof=0), private.std(ddof=0)])
print("Public:", sizes[0], "children; mean", round(public.mean(), 3), "; standard deviation", round(sds[0], 3))
print("Private:", sizes[1], "children; mean", round(private.mean(), 3), "; standard deviation", round(sds[1], 3))

# 2. Standard error of the municipality mean and of each stratum mean for three allocations
optimal = np.round(600 * weights * sds / (weights * sds).sum())
for name, allocation in (("Proportional", np.round(600 * weights)), ("Optimal", optimal), ("Equal", np.array([300, 300]))):
    variance = (weights ** 2 * sds ** 2 / allocation * (1 - allocation / sizes)).sum()
    se_h = sds / np.sqrt(allocation) * np.sqrt(1 - allocation / sizes)
    print(name, "allocation", allocation, "; SE overall", round(sqrt(variance), 4), "; SE public", round(se_h[0], 3), "; SE private", round(se_h[1], 3))

# 3. The proportional allocation by the formula: 600 times the weight of each stratum
exact = 600 * weights
print("Proportional allocation by the formula: public", round(exact[0], 1), "and private", round(exact[1], 1))

Public: 10851 children; mean 2.148 ; standard deviation 2.562
Private: 3448 children; mean 0.997 ; standard deviation 1.415
Proportional allocation [455. 145.] ; SE overall 0.0934 ; SE public 0.118 ; SE private 0.115
Optimal allocation [510.  90.] ; SE overall 0.0912 ; SE public 0.111 ; SE private 0.147
Equal allocation [300 300] ; SE overall 0.1123 ; SE public 0.146 ; SE private 0.078
Proportional allocation by the formula: public 455.3 and private 144.7


### 13.9 Sampling schools instead of children

*Can the survey visit only some of the schools?*

In [9]:
# 1. 10,000 samples of 20 schools (equal probability) with 30 children each
schools = np.array(sorted(df["school"].unique()))
rows_of = {}
for school in schools:
    rows_of[school] = np.where(df["school"].values == school)[0]
rng = np.random.default_rng(13)
school_means = []
for repetition in range(10000):
    chosen = rng.choice(schools, 20, replace=False)
    children = np.concatenate([rng.choice(rows_of[school], 30, replace=False) for school in chosen])
    school_means.append(dmft[children].mean())
school_means = np.array(school_means)
deff = (school_means.std() / srs_se) ** 2
print("Schools: standard deviation of the means", round(school_means.std(), 4), "; mean of the means", round(school_means.mean(), 3), "; design effect", round(deff, 2), "; effective sample size", round(600 / deff))

# 2. Intracluster correlation (analysis of variance) and the rule of thumb for 30 children per school
by_school = df.groupby("school")["dmft"].agg(["mean", "size"])
between = ((by_school["mean"] - mu) ** 2 * by_school["size"]).sum() / (len(schools) - 1)
within = ((df["dmft"] - df.groupby("school")["dmft"].transform("mean")) ** 2).sum() / (N - len(schools))
m0 = (N - (by_school["size"] ** 2).sum() / N) / (len(schools) - 1)
rho = (between - within) / (between + (m0 - 1) * within)
print("Intracluster correlation:", round(rho, 3), "; rule of thumb 1 + (30 - 1) rho:", round(1 + 29 * rho, 2))

# 3. Schools chosen with probability proportional to size (10,000 samples)
school_sizes = by_school["size"].reindex(schools).values
print("Mean of the 48 school means:", round(by_school["mean"].mean(), 3), "; population child mean:", round(mu, 3))
rng = np.random.default_rng(13)
pps_means = []
for repetition in range(10000):
    chosen = rng.choice(schools, 20, replace=True, p=school_sizes / school_sizes.sum())
    children = np.concatenate([rng.choice(rows_of[school], 30, replace=False) for school in chosen])
    pps_means.append(dmft[children].mean())
print("Proportional to size: mean of the means", round(np.mean(pps_means), 3), "; standard deviation", round(np.std(pps_means), 3), "; design effect", round((np.std(pps_means) / srs_se) ** 2, 2))

Schools: standard deviation of the means 0.1391 ; mean of the means 1.847 ; design effect 2.12 ; effective sample size 283
Intracluster correlation: 0.066 ; rule of thumb 1 + (30 - 1) rho: 2.91
Mean of the 48 school means: 1.846 ; population child mean: 1.87


Proportional to size: mean of the means 1.871 ; standard deviation 0.166 ; design effect 3.04


### 13.10 When there is no list of children

*When there is no list of children, what can be done?*

In [10]:
# 1. Quota sample: the first 150 private and the first 450 public children of the list
private_rows = np.where(df["school_type"].values == "private")[0]
public_rows = np.where(df["school_type"].values == "public")[0]
quota = np.concatenate([dmft[private_rows[:150]], dmft[public_rows[:450]]])
print("Quota sample: mean", round(quota.mean(), 3), "; free of decay:", round(100 * (quota == 0).mean(), 1), "%; schools:", df.iloc[np.concatenate([private_rows[:150], public_rows[:450]])]["school"].nunique())

Quota sample: mean 1.718 ; free of decay: 38.3 %; schools: 3


### 13.11 How many children must be examined?

*How many children must be examined?*

In [11]:
# 1. Sample size for the mean (margin 0.15), without and with the finite population correction
z = 1.959964
n0 = (z * sigma / 0.15) ** 2
n_corrected = n0 / (1 + (n0 - 1) / N)
print("Mean:", round(n0, 1), "; with the correction", round(n_corrected, 1))

# 2. Sample size for the share free of decay (margin 0.03) and for p = 0.5
p_free = (dmft == 0).mean()
n_share = z ** 2 * p_free * (1 - p_free) / 0.03 ** 2
print("Share:", round(n_share, 1), "; p = 0.5:", round(z ** 2 * 0.25 / 0.03 ** 2, 1))

# 3. Stratified designs (proportional and optimal), margin 0.15, without and with the finite population correction
n_proportional = (z / 0.15) ** 2 * (weights * sds ** 2).sum()
n_optimal = (z / 0.15) ** 2 * (weights * sds).sum() ** 2
n_proportional_c = n_proportional / (1 + n_proportional / N)
n_optimal_c = (weights * sds).sum() ** 2 / (0.15 ** 2 / z ** 2 + (weights * sds ** 2).sum() / N)
print("Without the correction: simple random", int(np.ceil(n0)), "; proportional", int(np.ceil(n_proportional)), "; optimal", int(np.ceil(n_optimal)))
print("With the correction: simple random", int(np.ceil(n_corrected)), "; proportional", int(np.ceil(n_proportional_c)), "; optimal", int(np.ceil(n_optimal_c)))

# 4. Margin of error with 600 children: simple random sample and 20 schools of 30
print("Margin of error: simple random sample", round(z * srs_se, 3), "; schools", round(z * school_means.std(), 3))

# 5. Visit all 48 schools and examine 20 or 30 children in each (school means weighted by school size)
school_sd = df.groupby("school")["dmft"].std(ddof=0).values
school_n = df.groupby("school").size().values
for m in (20, 30):
    variance_m = ((school_n / N) ** 2 * school_sd ** 2 / m * (1 - m / school_n)).sum()
    print("All 48 schools,", m, "children in each:", 48 * m, "children; margin", round(z * sqrt(variance_m), 3))

Mean: 974.0 ; with the correction 912.0
Share: 980.2 ; p = 0.5: 1067.1
Without the correction: simple random 975 ; proportional 933 ; optimal 892
With the correction: simple random 912 ; proportional 876 ; optimal 838
Margin of error: simple random sample 0.187 ; schools 0.273
All 48 schools, 20 children in each: 960 children; margin 0.145
All 48 schools, 30 children in each: 1440 children; margin 0.117


## Part B. Exercise solutions

### 13.12 Exercise 1. The first 600 children

In the first 600 children of the list the mean dmft is 1.295 and 264 children (44.0%) are free of decay; in the whole list the mean is 1.870 and 35.7% are free of decay. The first 600 come from three schools, S01, S02, and S03, all private; private-school children have less decay (mean 1.00) than public-school children (2.15), and the sample leaves out the 45 other schools. The bias is systematic: a larger sample from the same three schools would not remove it.

In [12]:
# 1. First 600 children and the whole list
print("First 600: mean", round(dmft[:600].mean(), 3), "; free of decay", round(100 * (dmft[:600] == 0).mean(), 1), "%; schools", list(df["school"].iloc[:600].unique()))
print("Whole list: mean", round(mu, 3), "; free of decay", round(100 * (dmft == 0).mean(), 1), "%")

First 600: mean 1.295 ; free of decay 44.0 %; schools ['S01', 'S02', 'S03']
Whole list: mean 1.87 ; free of decay 35.7 %


### 13.12 Exercise 2. One simple random sample

With seed 13 the sample of 600 has a mean of 1.847, a standard deviation of 2.373, and a standard error of $2.373/\sqrt{600}=0.0969$. The mean estimates the municipality mean; the standard deviation describes how the dmft varies among the 600 children; the standard error describes how much the sample mean would vary from sample to sample.

In [13]:
# 1. The sample of 600 with seed 13
print("Mean:", round(x.mean(), 3), "; standard deviation:", round(x.std(ddof=1), 3), "; standard error:", round(x.std(ddof=1) / sqrt(600), 4))

Mean: 1.847 ; standard deviation: 2.373 ; standard error: 0.0969


### 13.12 Exercise 3. Many simple random samples

The 2,000 means have a mean of 1.873 and a standard deviation of 0.0930. The standard error $\sigma/\sqrt{600}$ is $2.389/24.49=0.0975$, and corrected for finite population it is $0.0975\times0.979=0.0954$. The observed 0.0930 is close to both; the correction brings the formula slightly nearer.

In [14]:
# 1. Means of 2,000 samples against the standard error formulas
print("Mean of the means:", round(means600.mean(), 3), "; standard deviation:", round(means600.std(), 4), "; sigma/sqrt(n):", round(sigma / sqrt(600), 4), "; with the correction:", round(srs_se, 4))

Mean of the means: 1.873 ; standard deviation: 0.093 ; sigma/sqrt(n): 0.0975 ; with the correction: 0.0954


### 13.12 Exercise 4. Dividing by $n$ or by $n-1$

The population variance is 5.705. In 5,000 samples of 10 children the variance with $n$ averages 5.133, below 5.705, and the variance with $n-1$ averages 5.704, close to it. Dividing by $n$ underestimates the variance on average, because the deviations are measured from the sample mean.

In [15]:
# 1. Variances in 5,000 samples of 10 children
print("Population variance:", round(sigma ** 2, 3), "; dividing by n:", round(np.mean(variances_n), 3), "; dividing by n - 1:", round(np.mean(variances_n1), 3))

Population variance: 5.705 ; dividing by n: 5.133 ; dividing by n - 1: 5.704


### 13.12 Exercise 5. Skewness of the means

The skewness of the 2,000 means is 0.84 for samples of 5 children, 0.41 for 30, and 0.05 for 600. The normal curve describes the means well only with 600 children: 30 children are not enough for a variable as skewed as dmft.

In [16]:
# 1. Skewness of the means for each sample size
for n_children in (5, 30, 600):
    rng = np.random.default_rng(13)
    sample_means = []
    for repetition in range(2000):
        sample_means.append(dmft[rng.choice(N, n_children, replace=False)].mean())
    print("Samples of", n_children, ": skewness", round(stats.skew(sample_means), 2))

Samples of 5 : skewness 0.84
Samples of 30 : skewness 0.41
Samples of 600 : skewness 0.05


### 13.12 Exercise 6. Confidence interval for the mean

With $t_{0.975,599}=1.964$, the interval is $1.847\pm1.964\times0.0969=[1.656,\ 2.037]$. It contains the population mean, 1.870. In 2,000 samples of 600 children, 95.5% of the intervals contain it.

In [17]:
# 1. Interval for the sample of 600 and coverage in 2,000 samples
print("95% CI:", round(low, 3), "to", round(high, 3), "; contains", mu.round(3), ":", low <= mu <= high, "; coverage in 2,000 samples:", round(100 * hits / 2000, 1), "%")

95% CI: 1.656 to 2.037 ; contains 1.87 : True ; coverage in 2,000 samples: 95.5 %


### 13.12 Exercise 7. Wald and Wilson intervals

The share free of decay is 229/600 = 38.2%: Wald $[34.3\%,\ 42.1\%]$, Wilson $[34.4\%,\ 42.1\%]$. The share with dmft of 8 or more is 27/600 = 4.5%: Wald $[2.8\%,\ 6.2\%]$, Wilson $[3.1\%,\ 6.5\%]$. The difference matters more for severe decay: the share is near 0, where the Wald interval, which is symmetric around 4.5%, can be too narrow and can reach values that the data make implausible, while the Wilson interval is asymmetric and keeps away from 0.

In [18]:
# 1. Wald and Wilson intervals for both shares
for label, count in (("Share free of decay:", int((x == 0).sum())), ("Severe:", int((x >= 8).sum()))):
    share = count / 600
    wald = z * sqrt(share * (1 - share) / 600)
    centre = (share + z ** 2 / 1200) / (1 + z ** 2 / 600)
    half = z * sqrt(share * (1 - share) / 600 + z ** 2 / 1440000) / (1 + z ** 2 / 600)
    print(label, round(100 * share, 1), "%; Wald", round(100 * (share - wald), 1), "to", round(100 * (share + wald), 1), "; Wilson", round(100 * (centre - half), 1), "to", round(100 * (centre + half), 1))

Share free of decay: 38.2 %; Wald 34.3 to 42.1 ; Wilson 34.4 to 42.1
Severe: 4.5 %; Wald 2.8 to 6.2 ; Wilson 3.1 to 6.5


### 13.12 Exercise 8. Allocations

With proportional allocation (455 public, 145 private) the standard error of the municipality mean is 0.0934, with optimal allocation (510, 90) it is 0.0912, and with equal allocation (300, 300) it is 0.1123. The standard error of the stratum means is 0.118 (public) and 0.115 (private) with proportional allocation, 0.111 and 0.147 with optimal, and 0.146 and 0.078 with equal. Optimal allocation is best for the whole municipality; equal allocation is best for the private stratum.

In [19]:
# 1. Standard errors for the three allocations
for name, allocation in (("Proportional", np.round(600 * weights)), ("Optimal", optimal), ("Equal", np.array([300, 300]))):
    variance = (weights ** 2 * sds ** 2 / allocation * (1 - allocation / sizes)).sum()
    se_h = sds / np.sqrt(allocation) * np.sqrt(1 - allocation / sizes)
    print(name, "allocation", allocation, "; SE overall", round(sqrt(variance), 4), "; SE public", round(se_h[0], 3), "; SE private", round(se_h[1], 3))

Proportional allocation [455. 145.] ; SE overall 0.0934 ; SE public 0.118 ; SE private 0.115
Optimal allocation [510.  90.] ; SE overall 0.0912 ; SE public 0.111 ; SE private 0.147
Equal allocation [300 300] ; SE overall 0.1123 ; SE public 0.146 ; SE private 0.078


### 13.12 Exercise 9. Sampling schools

In 10,000 samples of 20 schools with 30 children each, the standard deviation of the means is 0.139. Against the simple random sample (standard error 0.0954), the design effect is $(0.139/0.0954)^{2}=2.12$, and the effective sample size is about $600/2.12=283$ children.

In [20]:
# 1. Design effect and effective sample size of 20 schools of 30 children
print("Standard deviation of the means:", round(school_means.std(), 4), "; design effect:", round(deff, 2), "; effective sample size:", round(600 / deff))

Standard deviation of the means: 0.1391 ; design effect: 2.12 ; effective sample size: 283


### 13.12 Exercise 10. A quota sample

The first 150 private and the first 450 public children give a mean of 1.718 and 38.3% free of decay, against 1.870 and 35.7% in the list. The quotas fix the share of each type of school, but the children are not drawn at random: they come from only three schools. A quota sample can be biased even when the quotas are right.

In [21]:
# 1. The quota sample
print("Mean:", round(quota.mean(), 3), "; free of decay:", round(100 * (quota == 0).mean(), 1), "%")

Mean: 1.718 ; free of decay: 38.3 %


### 13.12 Exercise 11. Sample sizes

For the mean with $E=0.20$ and $\sigma=2.389$: $n=(1.96\times2.389/0.20)^{2}=547.9$, or 548 children; with the finite population correction, 527.7, or 528. For the share free of decay with $E=0.04$ and $p=0.357$: $n=1.96^{2}\times0.357\times0.643/0.04^{2}=551.3$, or 552; with the correction, 530.9, or 531.

In [22]:
# 1. Sample sizes for the mean (margin 0.20) and the share (margin 0.04)
n_mean = (z * sigma / 0.20) ** 2
n_share2 = z ** 2 * p_free * (1 - p_free) / 0.04 ** 2
print("Mean:", round(n_mean, 1), "; with the correction", round(n_mean / (1 + (n_mean - 1) / N), 1), "; share:", round(n_share2, 1), "; with the correction", round(n_share2 / (1 + (n_share2 - 1) / N), 1))

Mean: 547.9 ; with the correction 527.7 ; share: 551.3 ; with the correction 530.9


### 13.12 Exercise 12. Two designs with 800 children

With 800 children chosen at random, the standard error is $2.389/\sqrt{800}\times\sqrt{(14{,}299-800)/14{,}298}=0.0821$, and the margin of error is $\pm0.161$. With 20 schools of 40 children and a design effect of 2.3, the standard error is $0.0821\times\sqrt{2.3}=0.1245$ and the margin is $\pm0.244$. The simple random sample gives the smaller margin; the school design is cheaper per child examined, and its extra margin is the price.

In [23]:
# 1. Margins of error of two designs with 800 children
se800 = sigma / sqrt(800) * sqrt((N - 800) / (N - 1))
print("Margin: 800 at random", round(z * se800, 3), "; 20 schools of 40 (design effect 2.3)", round(z * se800 * sqrt(2.3), 3))

Margin: 800 at random 0.161 ; 20 schools of 40 (design effect 2.3) 0.244


---
*Synthetic data (seed 13, `generate_data.py`). Teaching values only; nothing here is an estimate for a real municipality.*